# K-Nearest Neighbors (KNN)
## Real-world scenario: Classifying a flower species

A botanist measures a flower's **petal length** and **petal width** and wants to know its **species**. KNN classifies a new flower by looking at the *k* closest known flowers and taking a majority vote - a simple, intuitive algorithm.

### Step 1 - Import the libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

np.random.seed(42)

### Step 2 - Create a small, realistic dataset
Three flower species with different typical petal sizes.

In [ ]:
n_each = 25
# Setosa: small petals | Versicolor: medium | Virginica: large
setosa     = np.column_stack([np.random.normal(1.5, 0.2, n_each),
                              np.random.normal(0.3, 0.1, n_each)])
versicolor = np.column_stack([np.random.normal(4.3, 0.3, n_each),
                              np.random.normal(1.3, 0.2, n_each)])
virginica  = np.column_stack([np.random.normal(5.8, 0.3, n_each),
                              np.random.normal(2.1, 0.2, n_each)])

data = np.vstack([setosa, versicolor, virginica])
labels = ['setosa'] * n_each + ['versicolor'] * n_each + ['virginica'] * n_each

df = pd.DataFrame(data, columns=['petal_length', 'petal_width'])
df['species'] = labels

# Shuffle + add messy data
df = df.sample(frac=1, random_state=1).reset_index(drop=True)
df.loc[4, 'petal_length'] = np.nan
df = pd.concat([df, df.iloc[[0]]], ignore_index=True)
df.head()

### Step 3 - Explore the data

In [ ]:
print('Shape:', df.shape)
print('\nMissing:\n', df.isnull().sum())
print('\nDuplicates:', df.duplicated().sum())
print('\nSpecies counts:\n', df['species'].value_counts())

### Step 4 - Clean the data

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)
df['petal_length'] = df['petal_length'].fillna(df['petal_length'].median())
print('Missing after cleaning:', df.isnull().sum().sum())

### Step 5 - Features (X) and target (y)

In [ ]:
X = df[['petal_length', 'petal_width']]
y = df['species']

### Step 6 - Train / test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

### Step 7 - Scale the features
KNN measures distances, so features must be on the same scale.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

### Step 8 - Train the KNN model
`n_neighbors=3` means each prediction looks at the 3 nearest flowers.

In [ ]:
model = KNeighborsClassifier(n_neighbors=3)
model.fit(X_train_scaled, y_train)

### Step 9 - Evaluate

In [ ]:
y_pred = model.predict(X_test_scaled)
print('Accuracy:', round(accuracy_score(y_test, y_pred), 3))
print('\nConfusion matrix:\n', confusion_matrix(y_test, y_pred))
print('\nReport:\n', classification_report(y_test, y_pred, zero_division=0))

### Step 10 - Predict a new flower
Petal length 4.0, petal width 1.2:

In [ ]:
new_flower = pd.DataFrame({'petal_length': [4.0], 'petal_width': [1.2]})
new_scaled = scaler.transform(new_flower)
print('Predicted species:', model.predict(new_scaled)[0])